# CSoT'26 - ML in Astronomy - Week 5: VLM Audit + CNN/CLIP/VLM Capstone (Solution)

**Goal:** Prompt a **vision-language model** to interpret Euclid cutouts, catch it hallucinating, then put three tools on trial on the *same* images - the **Week-3 CNN**, the **Week-4 CLIP** ranker, and this week's **VLM**. Finish with a short capstone reflection.

This is the reference solution. Each cell mirrors the starter, with the code filled in and a short note on *why*. Try the starter yourself before reading this. **Use a GPU runtime.**

## Step 0 - Install libraries and pick a device

In [ ]:
# Run once per Colab session. qwen-vl-utils + accelerate support the Qwen2-VL loader.
!pip install -q datasets transformers accelerate qwen-vl-utils scikit-learn

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

## Step 1 - Reload the Euclid test set and take a stratified sample

VLM calls are slow, so we work on **30-50** cutouts. Make the sample **stratified**: include real lenses (label 1) *and* hard negatives (label 0) so precision/recall are meaningful. Keep images, labels, and ids aligned.

In [ ]:
from datasets import load_dataset

ds_test = load_dataset('mwalmsley/euclid_strong_lens_expert_judges', 'classification', split='test')
y_all = np.array(ds_test['label'])

# Stratified 40: 20 real lenses + 20 negatives, so precision/recall are meaningful
# on such a small sample. A pure random draw would be almost all negatives.
rng = np.random.default_rng(42)
pos = rng.choice(np.where(y_all == 1)[0], size=20, replace=False)
neg = rng.choice(np.where(y_all == 0)[0], size=20, replace=False)
idx = np.concatenate([pos, neg]); rng.shuffle(idx)
images = [ds_test[int(i)]['image'].convert('RGB') for i in idx]
labels = y_all[idx]
print('sample size:', len(images), 'positives:', int(labels.sum()))

## Step 2 (optional) - Reference captions

Peek at 2-3 image+caption pairs from `Arbie333/gravitational_lensing` to see what a *good* human description of a lens sounds like - a benchmark for judging the VLM's evidence.

In [ ]:
# A small set of human-written lens captions: a yardstick for the VLM's EVIDENCE.
ref = load_dataset('Arbie333/gravitational_lensing', split='train')
print(ref)
cap_field = [c for c in ref.column_names if c != 'image'][0]
for i in range(2):
    plt.imshow(ref[i]['image']); plt.axis('off'); plt.show()
    print('caption:', ref[i][cap_field])

## Step 3 - Load the VLM and define the fixed prompt

Load `Qwen/Qwen2-VL-2B-Instruct` in half precision. Use the **fixed** prompt from [`01`](../01-vlm-prompting-for-science.md) for *every* image - consistency is what makes the results comparable and reproducible.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

# float16 + device_map keep the 2B model within the T4's memory.
model_id = 'Qwen/Qwen2-VL-2B-Instruct'
vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map='auto')
processor = AutoProcessor.from_pretrained(model_id)

# One frozen prompt reused for every image = a controlled, reproducible experiment.
PROMPT = (
    'You are assisting an astronomer reviewing telescope cutouts for strong gravitational lensing.\n\n'
    'Look at this image. Strong lensing shows clear arcs, partial Einstein rings, or multiple images '
    'of a background source.\n\n'
    'Answer in this format:\n'
    'VERDICT: yes | no | uncertain\n'
    'EVIDENCE: one or two sentences describing what you see (or do not see)\n'
    'CAUTION: one sentence about what could be confused with lensing (e.g. spiral arms, mergers)')

## Step 4 - Run the VLM on every sampled image

Loop over the sample. For each image build the chat messages, generate with **low temperature / greedy decoding**, and store the raw text. One image per call keeps you off the T4 memory limit.

In [ ]:
# do_sample=False (greedy) makes the answers as repeatable as the model allows.
raw_answers = []
for img in images:
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': img}, {'type': 'text', 'text': PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors='pt').to(vlm.device)
    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=128, do_sample=False)
    reply = processor.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    raw_answers.append(reply)
print(raw_answers[0])

## Step 5 - Parse verdicts and score precision/recall

Parse the `VERDICT:` line, map `yes/no/uncertain` to a binary prediction, and compute precision/recall/F1 vs the true labels. **Document** how you treat `uncertain` (here: count as 'not a lens'; you could instead route it to human review).

In [ ]:
# Robust parsing: always return something, even if the model ignores the format.
def parse_verdict(text):
    for line in text.splitlines():
        if line.strip().upper().startswith('VERDICT'):
            v = line.split(':', 1)[1].strip().lower()
            if 'yes' in v: return 'yes'
            if 'uncertain' in v: return 'uncertain'
            if 'no' in v: return 'no'
    return 'uncertain'

verdicts = [parse_verdict(t) for t in raw_answers]
# Decision: treat 'uncertain' as 'not a lens' (conservative). Document this choice.
vlm_pred = np.array([1 if v == 'yes' else 0 for v in verdicts])
from sklearn.metrics import classification_report
print('verdict counts:', {v: verdicts.count(v) for v in set(verdicts)})
print(classification_report(labels, vlm_pred, target_names=['not lens', 'lens']))

## Step 6 - Hallucination hunt

Find **at least two** cutouts where the VLM said `yes` but the true label is `0` (and your eyes agree it's not a lens). Show the image alongside the model's own `EVIDENCE` quote - these are your documented hallucinations. (See [`02`](../02-hallucination-and-human-in-the-loop.md).)

In [ ]:
# False positives with confident evidence = hallucinations. Show image + the quote.
halluc = [i for i in range(len(images)) if vlm_pred[i] == 1 and labels[i] == 0]
print(f'{len(halluc)} confident false positives found')
for i in halluc[:2]:
    plt.imshow(images[i]); plt.axis('off'); plt.title('true label: not a lens'); plt.show()
    print('VLM said:\n', raw_answers[i], '\n')

## Step 7 - Reload the Week-3 CNN and run it on the SAME images

Rebuild `GalaxyCNN`, load `galaxy_model.pth`, and predict on the sample (resize to the CNN's `64x64`). Expect **poor relevance** - it has no lens class and was trained on SDSS Galaxy Zoo, not Euclid. That's the wrong-task + domain-shift lesson ([`04`](../04-comparing-cnn-clip-and-vlm.md)), not a bug.

In [ ]:
import torch.nn as nn
from torchvision import transforms

# Exact Week-3 architecture, so the saved state_dict loads cleanly.
class GalaxyCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * 16 * 16, 128), nn.ReLU(), nn.Linear(128, num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

# NOTE: use YOUR Week-3 train_ds.classes order here.
CLASSES = ['elliptical', 'spiral', 'lenticular']
cnn = GalaxyCNN(num_classes=len(CLASSES)).to(device)
# Retrain in Week-3 or mount Drive if galaxy_model.pth isn't in the session.
cnn.load_state_dict(torch.load('galaxy_model.pth', map_location=device)); cnn.eval()

# The CNN expects 64x64 RGB (Week-1 pipeline). The point: no 'lens' output exists.
tf = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
cnn_pred = []
for img in images:
    x = tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        cnn_pred.append(CLASSES[cnn(x).argmax(1).item()])
print('CNN can only ever answer with morphology classes:', set(cnn_pred))

## Step 8 - Week-4 CLIP scores on the SAME images

Reuse your Week-4 prompt bank and the `score = mean(lens sims) - mean(non-lens sims)` scorer on this sample, so all three tools see identical inputs.

In [ ]:
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
lens_prompts = ['a strong gravitational lens with an Einstein ring',
                'a gravitational lens arc next to a galaxy']
nonlens_prompts = ['a normal galaxy without gravitational lensing',
                   'a spiral galaxy with curved arms but no gravitational lens arc']
all_prompts = lens_prompts + nonlens_prompts
n_lens = len(lens_prompts)

t_in = clip_proc(text=all_prompts, return_tensors='pt', padding=True).to(device)
with torch.no_grad():
    t_emb = F.normalize(clip.get_text_features(**t_in), dim=-1)
i_in = clip_proc(images=images, return_tensors='pt').to(device)
with torch.no_grad():
    i_emb = F.normalize(clip.get_image_features(**i_in), dim=-1)
sims = (i_emb @ t_emb.T).cpu()
clip_scores = (sims[:, :n_lens].mean(1) - sims[:, n_lens:].mean(1)).numpy()
print('CLIP score range:', round(float(clip_scores.min()), 3), 'to', round(float(clip_scores.max()), 3))

## Step 9 - The three-way comparison figure

Put it all together: one row per image with the cutout and a caption stacking `true | CNN | CLIP | VLM`. Look for agreement on easy cases and **correlated** CLIP/VLM failures on spirals (the shared confusion from [`03`](../03-arcs-vs-spirals-confusion-physics.md)).

In [ ]:
# One figure, all three tools, identical inputs - the capstone artefact.
n = min(12, len(images))
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax, i in zip(axes.ravel(), range(n)):
    ax.imshow(images[i]); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"true={labels[i]} | CNN={cnn_pred[i]}\nCLIP={clip_scores[i]:.2f} | VLM={verdicts[i]}",
                 fontsize=8)
plt.tight_layout(); plt.show()

## (Optional) Hosted-API fallback

If you'd rather use a hosted VLM, replace Steps 3-4 with an API call (e.g. Google AI Studio's free-tier Gemini). Keep the **same** fixed prompt and the same parsing so the rest of the notebook is unchanged.

```python
# import google.generativeai as genai
# from google.colab import userdata
# genai.configure(api_key=userdata.get('GEMINI_KEY'))  # store in Colab secrets, never hard-code
# gmodel = genai.GenerativeModel('gemini-1.5-flash')
# raw_answers = [gmodel.generate_content([PROMPT, img]).text for img in images]
```

## Capstone Reflection *(example - write your own ~300 words)*

1. **Trust.** The VLM's recall on lenses is often moderate but its precision is dragged down by confident false positives on spiral- and ring-like negatives - so no, I would not run it unsupervised. It earns trust as a *describer* that flags candidates with human-readable evidence, and loses it whenever the EVIDENCE cites an arc I cannot see.
2. **The wrong tool.** The Week-3 CNN can only emit morphology classes and was trained on SDSS Galaxy Zoo, so on Euclid lens cutouts it is doubly out of domain. Its confident-looking predictions are meaningless here - a vivid reminder that a model is only valid on its training task and data.
3. **The pipeline.** I'd use CLIP (precision-tuned) to triage millions of cutouts into a shortlist, add VLM evidence to help experts prioritise, and keep humans (with spectroscopic follow-up) as the deciders. I'd set the threshold high enough to hand a human only a few hundred candidates per batch.
4. **Looking back.** I started unable to define a tensor; I can now train, evaluate, and - most importantly - *distrust* a model on purpose. Knowing when to call a human is the skill I'll keep.